# 01 · Exploratory Data Analysis — EMSCAD + Custom Indian set
Goals: understand **class imbalance**, **text length**, **vocabulary differences** (word clouds) and **missing-field patterns** between real and fake posts.

> Download `fake_job_postings.csv` from Kaggle into `data/raw/` first (see README).

In [ ]:
import sys, re
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_emscad_raw, emscad_to_common, load_custom_india
from src.features import extract_features_df, METADATA_COLUMNS
from src.preprocess import clean_text

sns.set_theme(style="whitegrid")
raw = load_emscad_raw()
df = emscad_to_common(raw)
print(raw.shape, "| columns:", list(raw.columns))

## 1. Class imbalance

In [ ]:
counts = raw["fraudulent"].value_counts().rename({0: "Real", 1: "Fake"})
ax = counts.plot.bar(color=["#16a34a", "#dc2626"], rot=0, figsize=(4, 3), title="Posts per class")
for p in ax.patches:
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2, p.get_height()), ha="center", va="bottom")
plt.show()
print(f"Fake ratio: {raw['fraudulent'].mean():.2%}")
print(f"'Always predict real' accuracy = {1 - raw['fraudulent'].mean():.2%}  <- why accuracy is misleading")

**Takeaway:** only ~5 % of posts are fake. A useless model that always says *real* gets ~95 % accuracy, so we evaluate with **recall / precision / F1 / PR-AUC on the fake class**, use **stratified splits**, and handle imbalance with **class weights / SMOTE**.

## 2. Text length

In [ ]:
df["n_words"] = df["text"].str.split().str.len()
df["label_name"] = df["label"].map({0: "Real", 1: "Fake"})
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
sns.histplot(data=df, x="n_words", hue="label_name", bins=60, stat="density", common_norm=False,
             log_scale=(True, False), ax=axes[0], palette={"Real": "#16a34a", "Fake": "#dc2626"})
axes[0].set_title("Words per post (log x)")
sns.boxplot(data=df, x="label_name", y="n_words", ax=axes[1], showfliers=False,
            palette={"Real": "#16a34a", "Fake": "#dc2626"})
axes[1].set_title("Words per post")
plt.tight_layout(); plt.show()
df.groupby("label_name")["n_words"].describe()

## 3. Word clouds — real vs fake

In [ ]:
df["clean_text"] = [clean_text(t) for t in df["text"]]

def show_words(texts, title, ax, color):
    try:
        from wordcloud import WordCloud
        wc = WordCloud(width=600, height=350, background_color="white", colormap=color,
                       max_words=120, collocations=False, random_state=42).generate(" ".join(texts))
        ax.imshow(wc); ax.axis("off")
    except ImportError:  # fallback: top-20 bar chart
        top = pd.Series(" ".join(texts).split()).value_counts().head(20)[::-1]
        top.plot.barh(ax=ax, color="#dc2626" if color == "Reds" else "#16a34a")
    ax.set_title(title)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
show_words(df.loc[df.label == 0, "clean_text"], "REAL posts", axes[0], "Greens")
show_words(df.loc[df.label == 1, "clean_text"], "FAKE posts", axes[1], "Reds")
plt.show()

In [ ]:
# Most *distinctive* bigrams: frequency in fake minus frequency in real
from sklearn.feature_extraction.text import CountVectorizer
cv = CountVectorizer(ngram_range=(2, 2), min_df=10, binary=True)
X = cv.fit_transform(df["clean_text"])
y = df["label"].values
fake_rate = np.asarray(X[y == 1].mean(axis=0)).ravel()
real_rate = np.asarray(X[y == 0].mean(axis=0)).ravel()
diff = pd.Series(fake_rate - real_rate, index=cv.get_feature_names_out())
pd.DataFrame({"more common in FAKE": diff.nlargest(15).index, "more common in REAL": diff.nsmallest(15).index})

## 4. Missing-field patterns

In [ ]:
fields = ["company_profile", "description", "requirements", "benefits", "salary_range", "department",
          "employment_type", "required_experience", "required_education", "industry", "function"]
miss = raw[fields].fillna("").astype(str).apply(lambda col: col.str.strip() == "")
miss_rate = miss.groupby(raw["fraudulent"]).mean().T.rename(columns={0: "Real", 1: "Fake"})
miss_rate["Fake - Real"] = miss_rate["Fake"] - miss_rate["Real"]
ax = miss_rate[["Real", "Fake"]].sort_values("Fake").plot.barh(figsize=(7, 5), color=["#16a34a", "#dc2626"],
                                                                title="Share of posts with field MISSING")
plt.show()
miss_rate.sort_values("Fake - Real", ascending=False).style.format("{:.1%}")

In [ ]:
# Binary columns in EMSCAD
for col in ["has_company_logo", "telecommuting", "has_questions"]:
    print(raw.groupby(col)["fraudulent"].mean().rename(f"fake rate by {col}").to_string(), "\n")
print("NOTE: has_company_logo is very predictive, but a user pasting text cannot provide it,")
print("so the model does NOT use it (avoids train/serve skew).")

## 5. Our metadata features by class

In [ ]:
feats = extract_features_df(df)
feats["label"] = df["label"].values
summary = feats.groupby("label")[METADATA_COLUMNS].mean().T.rename(columns={0: "Real (mean)", 1: "Fake (mean)"})
summary["ratio Fake/Real"] = summary["Fake (mean)"] / summary["Real (mean)"].replace(0, np.nan)
summary.style.format("{:.3f}")

Most Indian-specific flags (`asks_payment`, `offplatform_contact`, `free_email_domain`) are **rare in EMSCAD** — the dataset is US/UK 2012-14 and e-mails are anonymised. This motivates the separate **custom Indian evaluation set** and the **rule-based layer** in the app.

## 6. Custom Indian dataset

In [ ]:
india = load_custom_india()
print(india["label"].value_counts().rename({0: "Real", 1: "Fake"}).to_string())
print(india["source"].value_counts().to_string())
fi = extract_features_df(india); fi["label"] = india["label"].values
fi.groupby("label")[["asks_payment", "offplatform_contact", "free_email_domain", "no_interview",
                     "easy_money", "salary_too_high", "urgency_count"]].mean().T.rename(columns={0: "Real", 1: "Fake"})

## Summary
* Heavy imbalance (≈95/5) → stratified split, class weights / SMOTE, recall-oriented threshold, PR-AUC.
* Fake posts are shorter and more often miss company profile, requirements and benefits.
* Vocabulary differs (e.g. *data entry*, *work home*, *oil gas* vs *team*, *product*, *client*).
* EMSCAD lacks Indian scam signals → evaluate separately on the custom Indian set.